In [ ]:
"""
Malawi Diarrhoea Cases: SARIMA Forecast for 2026
==================================================
Workflow:
 1. Prepare time series
 2. Handle missing values
 3. Inspect for structural breaks (e.g. COVID-era reporting disruption)
 4. Check stationarity (non-seasonal AND seasonal)
 5. Validate model on a held-out test set (train/test split)
 6. Fit final SARIMA model on full data
 7. Diagnostic checks on residuals
 8. Forecast 12 months ahead
 9. Plot and export results

Required packages:
    pip install pandas numpy matplotlib statsmodels pmdarima --break-system-packages
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf
import pmdarima as pm
from sklearn.metrics import mean_absolute_percentage_error, mean_squared_error

# --------------------------------------------------------------------------
# Step 1: Load and prepare time series
# --------------------------------------------------------------------------
# diarrhoea_df should have a 'date' column and a 'cases' column.
# Adjust the file path / column names to match your actual data.
diarrhoea_df = pd.read_csv("diarrhoea_df.csv", parse_dates=["date"])
diarrhoea_df = diarrhoea_df.sort_values("date").set_index("date")

# Ensure monthly frequency (fills any missing months with NaN, to be
# interpolated in Step 2)
diarrhoea_ts = diarrhoea_df["cases"].asfreq("MS")

diarrhoea_ts.plot(figsize=(10, 4), title="Malawi Diarrhoea Cases: Full Observed Series")
plt.xlabel("Time")
plt.ylabel("Number of Cases")
plt.tight_layout()
plt.show()

# --------------------------------------------------------------------------
# Step 2: Handle missing values
# --------------------------------------------------------------------------
if diarrhoea_ts.isna().any():
    print("Missing values detected — interpolating.")
    diarrhoea_ts = diarrhoea_ts.interpolate(method="time")

# --------------------------------------------------------------------------
# Step 3: Check for structural breaks / outliers (e.g. COVID-era disruption)
# --------------------------------------------------------------------------
# Simple approach: flag points more than 3 standard deviations from a
# rolling seasonal decomposition residual. Inspect these manually.
decomp = seasonal_decompose(diarrhoea_ts, model="additive", period=12)
resid = decomp.resid.dropna()
outlier_threshold = 3 * resid.std()
outliers = resid[abs(resid) > outlier_threshold]

print("Potential outlier periods (residual > 3 SD):")
print(outliers)

decomp.plot()
plt.tight_layout()
plt.show()

# If a COVID-era disruption period is confirmed (e.g. 2020-04 to 2020-12
# had reporting gaps or reporting-system distortion rather than true
# incidence change), consider ONE of the following before proceeding:
#
# Option A — trim the disrupted period out entirely:
# diarrhoea_ts = diarrhoea_ts["2021-01-01":]
#
# Option B — replace flagged outlier points with interpolated values:
# diarrhoea_ts.loc[outliers.index] = np.nan
# diarrhoea_ts = diarrhoea_ts.interpolate(method="time")
#
# Decide based on visual inspection of the plots above and your knowledge
# of what happened to reporting in that period. Document whichever choice
# you make, since it materially affects the seasonal pattern learned below.

# --------------------------------------------------------------------------
# Step 4: Check stationarity — non-seasonal AND seasonal
# --------------------------------------------------------------------------
# ADF test: checks for a non-seasonal unit root only.
adf_result = adfuller(diarrhoea_ts.dropna())
print("\n--- Augmented Dickey-Fuller Test ---")
print(f"ADF Statistic: {adf_result[0]:.4f}")
print(f"p-value:       {adf_result[1]:.4f}")
print("Critical values:", adf_result[4])
# p > 0.05 => fail to reject null of non-stationarity (non-seasonal sense)

# pmdarima can suggest the number of differences needed (both non-seasonal
# and seasonal), similar to R's ndiffs()/nsdiffs()
n_diffs = pm.arima.ndiffs(diarrhoea_ts, test="adf")
n_seasonal_diffs = pm.arima.nsdiffs(diarrhoea_ts, m=12, test="ocsb")
print(f"\nSuggested non-seasonal differences (ndiffs): {n_diffs}")
print(f"Suggested seasonal differences (nsdiffs):     {n_seasonal_diffs}")

# --------------------------------------------------------------------------
# Step 5: Out-of-sample validation (train/test split)
# --------------------------------------------------------------------------
# Hold out the most recent 6 months as a test set to check forecast accuracy
# before trusting the final 12-month-ahead projection.
n_test = 6
train = diarrhoea_ts.iloc[:-n_test]
test = diarrhoea_ts.iloc[-n_test:]

fit_train = pm.auto_arima(
    train,
    seasonal=True,
    m=12,
    stepwise=False,       # more exhaustive search, mirrors R's stepwise = FALSE
    approximation=False,
    trace=True,
    error_action="ignore",
    suppress_warnings=True,
)

fc_test, ci_test = fit_train.predict(n_periods=n_test, return_conf_int=True)
fc_test = pd.Series(fc_test, index=test.index)

rmse = np.sqrt(mean_squared_error(test, fc_test))
mape = mean_absolute_percentage_error(test, fc_test) * 100

print(f"\n--- Out-of-sample accuracy (last {n_test} months held out) ---")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

plt.figure(figsize=(10, 4))
plt.plot(train.index, train, label="Train")
plt.plot(test.index, test, label="Actual", color="black")
plt.plot(fc_test.index, fc_test, label="Forecast", color="red")
plt.fill_between(test.index, ci_test[:, 0], ci_test[:, 1], color="red", alpha=0.2)
plt.title("Validation: Forecast vs Actual (Held-Out Period)")
plt.xlabel("Time")
plt.ylabel("Number of Cases")
plt.legend()
plt.tight_layout()
plt.show()

# Review the RMSE / MAPE above. If MAPE is very high (e.g. >30-40%),
# treat the final forecast with caution and say so in your report —
# especially given the series likely has fewer than 8 full seasonal
# cycles, which limits how reliably SARIMA can estimate seasonality.

# --------------------------------------------------------------------------
# Step 6: Fit final model on the FULL series
# --------------------------------------------------------------------------
fit = pm.auto_arima(
    diarrhoea_ts,
    seasonal=True,
    m=12,
    stepwise=False,
    approximation=False,
    trace=True,
    error_action="ignore",
    suppress_warnings=True,
)

print(fit.summary())

# --------------------------------------------------------------------------
# Step 7: Diagnostic checks
# --------------------------------------------------------------------------
fit.plot_diagnostics(figsize=(10, 8))
plt.tight_layout()
plt.show()

residuals = pd.Series(fit.resid())
lb_test = acorr_ljungbox(residuals, lags=[12], return_df=True)
print("\n--- Ljung-Box test on residuals ---")
print(lb_test)
# Want p > 0.05, indicating residuals are consistent with white noise
# (no leftover structure the model missed).

# --------------------------------------------------------------------------
# Step 8: Forecast next 12 periods
# --------------------------------------------------------------------------
n_forecast = 12
forecast_mean, conf_int_80 = fit.predict(n_periods=n_forecast, return_conf_int=True, alpha=0.20)
_, conf_int_95 = fit.predict(n_periods=n_forecast, return_conf_int=True, alpha=0.05)

forecast_index = pd.date_range(
    start=diarrhoea_ts.index[-1] + pd.DateOffset(months=1),
    periods=n_forecast,
    freq="MS",
)
forecast_mean = pd.Series(forecast_mean, index=forecast_index)

print("\n--- 12-Month Forecast ---")
print(forecast_mean)

# --------------------------------------------------------------------------
# Step 9: Plot final forecast
# --------------------------------------------------------------------------
last_obs_label = diarrhoea_ts.index[-1].strftime("%Y-%m")

plt.figure(figsize=(10, 5))
plt.plot(diarrhoea_ts.index, diarrhoea_ts, label="Observed")
plt.plot(forecast_mean.index, forecast_mean, label="Forecast", color="red")
plt.fill_between(forecast_index, conf_int_95[:, 0], conf_int_95[:, 1],
                  color="red", alpha=0.15, label="95% CI")
plt.fill_between(forecast_index, conf_int_80[:, 0], conf_int_80[:, 1],
                  color="red", alpha=0.25, label="80% CI")
plt.title(f"Malawi 12-Month Forecast of Diarrhoea Cases\n(from {last_obs_label} onward)")
plt.xlabel("Time")
plt.ylabel("Number of Cases")
plt.legend()
plt.tight_layout()
plt.show()

# --------------------------------------------------------------------------
# Optional: export forecast table for the report
# --------------------------------------------------------------------------
forecast_df = pd.DataFrame({
    "date": forecast_index,
    "forecast": forecast_mean.values,
    "lower_80": conf_int_80[:, 0],
    "upper_80": conf_int_80[:, 1],
    "lower_95": conf_int_95[:, 0],
    "upper_95": conf_int_95[:, 1],
})

forecast_df.to_csv("diarrhoea_forecast_2026.csv", index=False)
print("\nForecast exported to diarrhoea_forecast_2026.csv")